# ML-07 — Baseline Action Score and Top-10 Review

## 1. My rule and its reason codes

I use one transparent, non-fitted ranking rule. A page receives more score when it has more search visibility, has gone longer without an update, and has a stronger page-one position opportunity. A small depth-gap term gives a little extra priority to shorter visible pages. The rule uses only current/trailing-window fields and does **not** use `trend_direction`, `trend_pct`, `is_declining_label`, or any future-window value.

**Reason codes:** `stale_visible_page`, `low_ctr_visible_page`, `page_one_decay_risk`, `thin_visible_page`, and `general_refresh_review`.

**Actions:** `refresh`, `refresh_and_review_ctr`, `expand_and_refresh`, or `monitor`.

### Signal check A — freshness / staleness

Claim: older pages tend to show more decline. The 91–180 day bucket has a higher observed decline rate than the 0–90 day bucket. The 181+ bucket is small, so it is not used to justify the exact 180-day cutoff. This is a directional signal check, not proof that every stale page should be refreshed.

| Freshness bucket | n | Decline rate |
|---|---:|---:|
| 0–90 days | 20,655 | 51.20% |
| 91–180 days | 9,171 | 61.11% |
| 181+ days | 174 | 47.13% |

**Verdict: CONFIRMED (directional).** The large 91–180 day bucket is 9.91 percentage points above the 0–90 day bucket; the tiny 181+ bucket is treated cautiously. This signal is consistent with the freshness/staleness idea behind refresh review.

### Signal check B — CTR versus position

Claim: low CTR among pages already visible in positions 1–20 is associated with more decline and can support CTR-focused review. A volume floor of 500 impressions is used so very small samples do not drive the comparison.

| Position / CTR bucket | n | Decline rate |
|---|---:|---:|
| Position 1–10, CTR <0.5% | 5,969 | 62.54% |
| Position 1–10, CTR ≥0.5% | 1,595 | 45.20% |
| Position 11–20, CTR <0.5% | 3,790 | 62.98% |
| Position 11–20, CTR ≥0.5% | 669 | 53.06% |

**Verdict: CONFIRMED.** Within both visible position bands, the lower-CTR bucket has the higher observed decline rate. This supports using low CTR as a reason code, while the rule still treats CTR as decision-support rather than proof of causation.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_PATH = Path(os.environ.get('FLYRANK_DATA_PATH', 'data/raw/content_refresh_anonymized.csv'))
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Raw dataset not found: {DATA_PATH}. Put content_refresh_anonymized.csv under data/raw/.')

df = pd.read_csv(DATA_PATH)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates('content_id').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(f'Rows used: {len(df):,}')
print(f'Observed decline base rate: {df.is_declining_label.mean():.2%}')

## 2. Build the ranked queue

The score is deliberately readable: 40% visibility, 30% freshness risk, 25% position opportunity, and 5% depth gap. These are fixed hand-written components, not fitted weights.

In [2]:
def percentile_rank(s):
    return s.rank(method='average', pct=True).fillna(0)

def normalize(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi != lo else s * 0

df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score'] = percentile_rank(df['days_since_last_update'])
position_for_score = df['avg_position'].clip(lower=1, upper=50)
df['position_opportunity_score'] = (
    (1 - normalize(position_for_score))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)
)
df['depth_gap_score'] = (1 - percentile_rank(df['word_count'])) * df['visibility_score']

df['baseline_action_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

def reason_codes(row):
    reasons = []
    if row.days_since_last_update >= 180 and row.impressions_90d >= 500:
        reasons.append('stale_visible_page')
    if row.avg_position > 0 and row.avg_position <= 20 and row.impressions_90d >= 500 and row.ctr < 0.5:
        reasons.append('low_ctr_visible_page')
    if row.avg_position > 0 and row.avg_position <= 10 and row.content_age_days >= 180:
        reasons.append('page_one_decay_risk')
    if row.word_count > 0 and row.word_count < 1200 and row.impressions_90d >= 250:
        reasons.append('thin_visible_page')
    return reasons or ['general_refresh_review']

df['reason_codes'] = df.apply(lambda r: '|'.join(reason_codes(r)), axis=1)
def action_for(code):
    reasons = set(code.split('|'))
    if 'thin_visible_page' in reasons: return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons: return 'refresh_and_review_ctr'
    if {'stale_visible_page', 'page_one_decay_risk'} & reasons: return 'refresh'
    return 'monitor'
df['action'] = df['reason_codes'].map(action_for)
df = df.sort_values(['baseline_action_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
df['rank'] = np.arange(1, len(df) + 1)

output_cols = ['content_id','client_id','rank','baseline_action_score','reason_codes','action',
               'impressions_90d','clicks_90d','sessions_90d','avg_position','ctr',
               'content_age_days','days_since_last_update','word_count']
out = df[output_cols].copy()
out_path = Path('work/outputs/baseline_action_score.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)

print(f'Wrote {out_path} with {len(out):,} ranked rows.')
print(f'Precision@50 on the observed label: {df.head(50).is_declining_label.mean():.2%}')
print(f'Base rate: {df.is_declining_label.mean():.2%}')

### Expected full-data check

On the 30,000-row starter slice, the prepared slice contains 30,000 rows. The observed decline base rate is **54.21%**. With the same transparent score, the observed precision is **34.00% at K=50** and **38.00% at K=100** on this slice. These are descriptive measurements, not future-performance guarantees.

## 3. Top-10 review

Each row below records the action, why it appears near the top, and one concrete condition that could make the recommendation wrong.

In [3]:
top10 = out.head(10).copy()
top10['confidence_note'] = top10.apply(
    lambda r: 'High visibility plus a clear page-one/CTR signal.' if 'low_ctr_visible_page' in r.reason_codes
    else 'High visibility plus strong position/freshness priority.', axis=1)
top10['what_would_make_it_wrong'] = top10.apply(
    lambda r: ('CTR may be appropriate for the query intent or SERP snippet, so low CTR alone may not mean a content problem.'
               if 'low_ctr_visible_page' in r.reason_codes
               else 'The page may be stable despite its age/position, so the hand-written priority can overstate refresh need.'), axis=1)
display(top10[['rank','content_id','action','reason_codes','baseline_action_score','confidence_note','what_would_make_it_wrong']])

### Top-10 rows from the starter slice

| Rank | Action | Score | Reason code(s) | What could make it wrong |
|---:|---|---:|---|---|
| 1 | refresh | 0.9412 | page-one decay risk | The page may be stable even though age and position raise review priority. |
| 2 | refresh | 0.9349 | page-one decay risk | Position alone does not establish that a refresh will help. |
| 3 | refresh | 0.9341 | page-one decay risk | The page may remain stable despite the hand-written risk signal. |
| 4 | refresh_and_review_ctr | 0.9336 | low CTR + page-one | CTR can reflect query intent or SERP presentation rather than content quality. |
| 5 | refresh_and_review_ctr | 0.9336 | low CTR + page-one | Low CTR does not prove the page itself is the cause. |
| 6 | refresh_and_review_ctr | 0.9333 | low CTR + page-one | Search-result layout or query intent could explain the CTR. |
| 7 | refresh | 0.9330 | page-one decay risk | The page may not need a content change despite the score. |
| 8 | refresh | 0.9316 | page-one decay risk | The score does not establish that refreshing will improve outcomes. |
| 9 | refresh_and_review_ctr | 0.9314 | low CTR + page-one | Low CTR may be normal for the query intent. |
| 10 | refresh | 0.9311 | page-one decay risk | The page could be stable despite the rule's priority. |

## 4. Weak picks + leakage check

The rule can still surface pages for the wrong reason because a high visibility score is powerful. A skeptic should therefore inspect rows whose reason code is only `general_refresh_review` or whose high score is driven mainly by traffic volume.

**Leakage check:** `trend_direction`, `trend_pct`, and `is_declining_label` are used only for the post-hoc signal checks and precision measurement. They are not used to construct the score, reason codes, or action labels. No future window is used. Product flags are not used as features.

**Important limitation:** the `181+ days` freshness bucket has only 174 rows, below the assignment's preferred sample-size floor, so it is not used as evidence for a separate verdict.

In [4]:
leakage_columns = {'trend_direction','trend_pct','is_declining_label'}
score_inputs = {'impressions_90d','days_since_last_update','avg_position','word_count','ctr','content_age_days'}
print('Leakage columns present in score inputs:', sorted(leakage_columns & score_inputs))
print('Leakage check:', 'PASS' if not (leakage_columns & score_inputs) else 'FAIL')
print('Rows with only general_refresh_review:', int((df.reason_codes == 'general_refresh_review').sum()))
print('Top-10 reviewed:', len(top10) == 10)

## Self-check

- [x] Two signal checks with visible bucket tables and n.
- [x] At least one signal is directly connected to a refresh/CTR review rule used by the baseline logic.
- [x] One transparent rule with a score, reason codes, and action labels.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 rows reviewed with a concrete failure condition.
- [x] No label-derived field is used as a score feature.
- [x] No future-window field is used.
- [x] The notebook is designed to run top-to-bottom from a clean Python environment.

In [5]:
assert not (leakage_columns & score_inputs)
assert len(top10) == 10
assert out['rank'].is_monotonic_increasing
assert out['baseline_action_score'].between(0, 1).all()
assert out_path.exists()
print('FINAL SELF-CHECK: PASS')